# Elección del modelo: Qwen 2.5 3B-Instruct

**¿Por qué Qwen 2.5 3B?**
- Ya está en producción en NormaBot via Ollama para la etapa de grading
- Open-weight, totalmente compatible con LoRA y QLoRA
- Suficientemente ligero para entrenarlo en una GPU con ≥ 8 GB VRAM
- Muy buena capacidad semántica en español para su tamaño
- La tarea es clasificación binaria simple, no requiere un modelo más grande

**Parámetros:** ~3.000 millones  
**Cuantización durante entrenamiento:** 4-bit NF4 (QLoRA, `bitsandbytes`)  
**Adaptadores LoRA:** r=8, alpha=16, capas de atención (`q_proj`, `k_proj`, `v_proj`, `o_proj`)

> Entrenado localmente en **NVIDIA RTX 4070 Super (13 GB VRAM)**.  
> Compatible con cualquier GPU ≥ 8 GB VRAM (RTX 3080, A10G, etc.).

# Instalación de librerías y comprobación de entorno


In [ ]:
import sys
print(sys.executable)
print(sys.version)

In [ ]:
# Dependencias necesarias — instalar antes de ejecutar este notebook:
#
#   GPU NVIDIA (CUDA 12.4 — RTX 4070 Super / driver >= 525):
#     pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
#
#   Resto del stack:
#     pip install -r requirements/finetuning.txt

import torch
print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}", end="")
if torch.cuda.is_available():
    print(f" | GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB)")
else:
    print("\nSin GPU: el entrenamiento será muy lento. Se recomienda GPU con >= 8 GB VRAM.")

In [ ]:
import os
import torch
from pathlib import Path
import sys

# Asegura que el directorio de trabajo sea la raíz del repositorio
# (necesario si Jupyter se abrió desde src/finetuning/ en vez de desde la raíz)
_here = Path.cwd()
if not (_here / "requirements").exists():
    for _parent in _here.parents:
        if (_parent / "requirements").exists():
            os.chdir(_parent)
            break
    else:
        raise RuntimeError(
            f"No se encontró la raíz del repositorio desde {_here}.\n"
            "Ejecuta Jupyter desde la raíz del proyecto: jupyter lab"
        )

# Importar funciones auxiliares del módulo local
sys.path.insert(0, str(Path.cwd() / "src" / "finetuning"))
from functions import (  # noqa: E402
    GRADING_SYSTEM_PROMPT,
    check_gpu, load_grading_dataset, split_dataset,
    format_training_prompt, examples_to_hf_dataset,
    load_tokenizer, load_model_4bit, build_peft_model, get_training_args,
    run_training, save_adapter,
)

check_gpu()

In [ ]:
from pathlib import Path

# El CWD ya fue fijado a la raíz del repositorio en la celda anterior (env-check-05)

# Rutas locales (relativas a la raíz del repositorio)
DATASET_PATH = Path("data/processed/grading_dataset.jsonl")

LABEL_RELEVANTE    = "relevante"
LABEL_NO_RELEVANTE = "no relevante"
LABELS = [LABEL_RELEVANTE, LABEL_NO_RELEVANTE]

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

OUTPUT_DIR   = "src/finetuning/output/qwen-grader-lora"
ADAPTER_PATH = "src/finetuning/output/qwen-grader-lora/adapter_final"
MERGED_PATH  = "src/finetuning/output/qwen-grader-merged"

MAX_SEQ_LEN = 512

print(f"Directorio de trabajo: {Path.cwd()}")
print("Configuración:")
print(f"  Modelo base:    {MODEL_ID}")
print(f"  Dataset:        {DATASET_PATH}  (existe: {DATASET_PATH.exists()})")
print(f"  Output dir:     {OUTPUT_DIR}")
print(f"  Adapter path:   {ADAPTER_PATH}")
print(f"  Max seq len:    {MAX_SEQ_LEN}")

In [ ]:
all_data = load_grading_dataset(DATASET_PATH)

In [ ]:
train_data, val_data, test_data = split_dataset(all_data)

In [ ]:
tokenizer  = load_tokenizer(MODEL_ID)
base_model = load_model_4bit(MODEL_ID)

# Formato del prompt de entrenamiento

Para el fine-tuning usamos el **chat template nativo de Qwen 2.5** (formato `<|im_start|>`).
Cada ejemplo de entrenamiento tiene la forma:

```
<|im_start|>system
Eres un asistente especializado en normativa de IA...<|im_end|>
<|im_start|>user
Consulta: {query}

Documento: {document}

¿Es este documento relevante para responder la consulta?<|im_end|>
<|im_start|>assistant
{label}<|im_end|>
```

El modelo aprende a generar directamente `relevante` o `no relevante` como turno de assistant.


In [ ]:
sample_formatted = format_training_prompt(train_data[0], tokenizer)
print("Ejemplo de prompt de entrenamiento:")
print("-" * 65)
print(sample_formatted)
print("-" * 65)
print(f"Longitud: {len(sample_formatted)} chars")

In [ ]:
train_dataset = examples_to_hf_dataset(train_data, tokenizer)
val_dataset   = examples_to_hf_dataset(val_data,   tokenizer)

print(f"Dataset de entrenamiento: {len(train_dataset)} ejemplos")
print(f"Dataset de validacion:    {len(val_dataset)} ejemplos")

# Fine-tuning con QLoRA

**QLoRA** (Quantized Low-Rank Adaptation) combina:
- **Cuantización 4-bit NF4** para reducir el uso de VRAM
- **LoRA** para entrenar solo un subconjunto pequeño de parámetros

Con r=8 entrenamos aproximadamente **el 1-2% de los parámetros totales** del modelo,
lo que hace el fine-tuning viable en una GPU T4 (16 GB VRAM).

Usamos `SFTTrainer` de `trl` (Supervised Fine-Tuning Trainer), que gestiona automáticamente
el packing de secuencias, la máscara de padding y la pérdida solo sobre el turno de assistant.


In [ ]:
try:
    del base_model
    torch.cuda.empty_cache()
    print("Modelo baseline liberado de VRAM.")
except NameError:
    pass

model = load_model_4bit(MODEL_ID, for_training=True)

In [ ]:
model, lora_config = build_peft_model(model)

In [ ]:
training_args = get_training_args(OUTPUT_DIR)

In [ ]:
train_result = run_training(
    model, training_args, train_dataset, val_dataset, tokenizer, MAX_SEQ_LEN
)

# Guardado del modelo

Guardamos el **adaptador LoRA** en `src/finetuning/output/qwen-grader-lora/adapter_final/`.
El adaptador pesa solo unos pocos MB y es suficiente para inferencia si se carga junto al modelo base.

Para producción con Ollama, descomenta el bloque de merge para obtener el modelo completo
y convertirlo a GGUF.

In [ ]:
metadata = {
    "model_id":              MODEL_ID,
    "task":                  "rag_relevance_grading",
    "labels":                LABELS,
    "lora_r":                lora_config.r,
    "lora_alpha":            lora_config.lora_alpha,
    "train_size":            len(train_data),
    "val_size":              len(val_data),
    "test_size":             len(test_data),
    "grading_system_prompt": GRADING_SYSTEM_PROMPT,
}
save_adapter(model, tokenizer, ADAPTER_PATH, metadata)

# OPCIONAL: merge adaptador + modelo base para convertir a GGUF
# os.makedirs(MERGED_PATH, exist_ok=True)
# from peft import PeftModel
# base_for_merge = load_model_4bit(MODEL_ID)
# merged = PeftModel.from_pretrained(base_for_merge, ADAPTER_PATH)
# merged = merged.merge_and_unload()
# save_adapter(merged, tokenizer, MERGED_PATH, metadata)
# print("Siguiente paso: convertir a GGUF con llama.cpp y subir a Ollama.")